# qust operator and context performance comparison

[项目地址](https://baiguoname.github.io/qust/site) · [git地址](https://github.com/baiguoname/qust)


本节 对比常见算子和常见上下文的性能，重点放在日常研究里高频使用的计算路径。

对比范围：

| 类别 | 场景 |
| --- | --- |
| 基础聚合 | `mean/std/sum/count/min/max` |
| 分组聚合 | `group_by("ticker")` 下的多聚合、first/last value |
| 序列算子 | `shift`、`ffill` |
| rolling | `rolling_mean`、`rolling_std`、`rolling_sum`、`rolling_rank` |
| expanding + over | 每个 `ticker` 独立累计 `sum` 和 `mean` |
| rolling + over | 每个 `ticker` 独立 rolling mean/std/rank |
| batch/cross-section | `batch.rank().over("date")`、`batch.sort("close")` |
| 常见 TA | `RSI(14)`，Polars/DuckDB 没有内置等价时作为 qust-only 参考 |

所有基准场景都展开成 50 列宽表再跑一遍，单列算子不会只算一列。
图形使用 qust monitor bar。底层 bar 已支持同一 x 分类下多 y 序列并排显示，所以 Polars/qust/DuckDB 会用不同颜色展示。


## 阅读方式

读法：

1. `Scenario` 定义了算子、上下文和各引擎实现；
2. `summary` 表看每个场景的 median/best/rows；
3. 第一张图是不同引擎的 median 耗时并排柱；
4. 第二张图是不同引擎的 best 耗时并排柱；
5. 第三张图只展示 qust-only 的常见领域算子。

这里固定使用内存中的 `DataFrame`，只看算子和上下文本身的执行开销。


In [ ]:
import gc
import time
from dataclasses import dataclass
from typing import Callable

import qust as qs

import qust.future.future
import qust.future

from qust import col
from qust._polars import pl

try:
    import duckdb
    HAS_DUCKDB = True
except Exception as exc:
    duckdb = None
    HAS_DUCKDB = False
    DUCKDB_IMPORT_ERROR = repr(exc)
else:
    DUCKDB_IMPORT_ERROR = None

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(18)

DATA_PATH = "https://github.com/baiguoname/qust/blob/main/examples/data/data_kline3.parquet?raw=true"
BENCH_N_ROWS = 500_000
REPEAT = 5
WARMUP = 1
WIDE_COLS = 50


## 1. Benchmark 数据

读取GitHub K 线数据后，取前 50 万行作为 benchmark 数据，并额外生成：

- `date`：横截面上下文用；
- `ret`：收益类算子或后续扩展示例可用。

读取数据和派生 `date/ret` 不计入 benchmark 时间。


In [11]:
raw = pl.read_parquet(DATA_PATH)
data = (
    raw
    .head(BENCH_N_ROWS)
    .with_columns(
        pl.col("datetime").dt.date().alias("date"),
        (pl.col("close") / pl.col("open") - 1.0).alias("ret"),
    )
)

engine_status = pl.DataFrame({
    "engine": ["polars", "qust", "duckdb"],
    "available": [True, True, HAS_DUCKDB],
    "note": [
        "eager DataFrame API",
        "Expr.calc_data(data)",
        "import duckdb ok" if HAS_DUCKDB else f"skipped: {DUCKDB_IMPORT_ERROR}",
    ],
})

wide_sort_data = data.with_columns(
    *[pl.col("close").alias(f"close_w{i:02d}") for i in range(WIDE_COLS)]
)

print("raw shape:", raw.shape)
print("benchmark shape:", data.shape)
print("wide sort shape:", wide_sort_data.shape)
display(engine_status)
data.head(5)


raw shape: (610463, 8)
benchmark shape: (500000, 10)
wide sort shape: (500000, 60)


engine,available,note
str,bool,str
"""polars""",true,"""eager DataFrame API"""
"""qust""",true,"""Expr.calc_data(data)"""
"""duckdb""",false,"""skipped: ModuleNotFoundError(""…"


ticker,datetime,open,high,low,close,volume,is_finished,date,ret
str,datetime[ms],f64,f64,f64,f64,f64,bool,date,f64
"""au""",2022-07-02 00:01:00,390.160004,390.160004,390.059998,390.119995,81.0,true,2022-07-02,-0.000103
"""au""",2022-07-02 00:02:00.500,390.119995,390.140015,390.079987,390.140015,52.0,true,2022-07-02,0.000051
"""au""",2022-07-02 00:03:00,390.140015,390.200012,390.119995,390.200012,50.0,true,2022-07-02,0.000154
"""au""",2022-07-02 00:04:01,390.200012,390.220001,390.140015,390.160004,61.0,true,2022-07-02,-0.000103
"""au""",2022-07-02 00:05:00.500,390.140015,390.140015,390.079987,390.100006,41.0,true,2022-07-02,-0.000103


## 2. 场景定义：常见算子 + 常见上下文

字段说明：

| 字段 | 含义 |
| --- | --- |
| `family` | 算子大类 |
| `context` | 执行上下文 |
| `operator` | 具体算子 |
| `qust_fn` | qust 实现 |
| `polars_fn` | Polars 等价实现 |
| `duckdb_sql` | DuckDB 等价 SQL，没有等价时为空 |


In [12]:
@dataclass
class Scenario:
    name: str
    label: str
    family: str
    context: str
    operator: str
    description: str
    qust_fn: Callable[[], object]
    polars_fn: Callable[[], object] | None = None
    duckdb_sql: str | None = None

WIDE_COLS = 50

def wide_named_exprs(named_exprs, total: int = WIDE_COLS):
    named_exprs = list(named_exprs)
    if not named_exprs:
        raise ValueError("wide_named_exprs requires at least one expression")
    wide = []
    for i in range(total):
        base_name, expr = named_exprs[i % len(named_exprs)]
        suffix = i // len(named_exprs)
        alias = f"{base_name}_{suffix:02d}"
        wide.append((alias, expr))
    return wide

def q_select(named_exprs, keys=()):
    wide = wide_named_exprs(named_exprs)
    exprs = [expr.alias(alias) for alias, expr in wide]
    return col(*keys, *exprs).calc_data(data)

def p_select(named_exprs, keys=()):
    wide = wide_named_exprs(named_exprs)
    exprs = [expr.alias(alias) for alias, expr in wide]
    return data.select(*keys, *exprs)

def q_groupby(named_exprs):
    wide = wide_named_exprs(named_exprs)
    exprs = [expr.alias(alias) for alias, expr in wide]
    return col(*exprs).group_by("ticker").calc_data(data)

def p_groupby(named_exprs):
    wide = wide_named_exprs(named_exprs)
    exprs = [expr.alias(alias) for alias, expr in wide]
    return data.group_by("ticker").agg(*exprs)

def q_over(named_exprs, by: str, keys=()):
    wide = wide_named_exprs(named_exprs)
    aliases = [alias for alias, _ in wide]
    exprs = [expr.alias(alias) for alias, expr in wide]
    return (
        col
        .with_cols(*exprs)
        .over(by)
        .select(*keys, *aliases)
        .calc_data(data)
    )

def p_over(named_exprs, by: str, keys=()):
    wide = wide_named_exprs(named_exprs)
    aliases = [alias for alias, _ in wide]
    exprs = [expr.over(by).alias(alias) for alias, expr in wide]
    return data.with_columns(*exprs).select(*keys, *aliases)

def duck_select(named_sql_exprs, *, table="bench_data", keys=(), group_by=None, order_by=None):
    wide = wide_named_exprs(named_sql_exprs)
    select_parts = [*keys, *[f"{sql} {alias}" for alias, sql in wide]]
    query = f"select {', '.join(select_parts)} from {table}"
    if group_by is not None:
        query += f" group by {group_by}"
    if order_by is not None:
        query += f" order by {order_by}"
    return query

def q_global_basic_agg():
    return q_select([
        ("mean_close", col("close").mean()),
        ("std_close", col("close").std()),
        ("sum_volume", col("volume").sum()),
        ("count_close", col("close").count()),
    ])

def p_global_basic_agg():
    return p_select([
        ("mean_close", pl.col("close").mean()),
        ("std_close", pl.col("close").std()),
        ("sum_volume", pl.col("volume").sum()),
        ("count_close", pl.col("close").count()),
    ])

def q_global_min_max():
    return q_select([
        ("min_low", col("low").min()),
        ("max_high", col("high").max()),
    ])

def p_global_min_max():
    return p_select([
        ("min_low", pl.col("low").min()),
        ("max_high", pl.col("high").max()),
    ])

def q_groupby_basic_agg():
    return q_groupby([
        ("mean_close", col("close").mean()),
        ("std_close", col("close").std()),
        ("sum_volume", col("volume").sum()),
        ("count_close", col("close").count()),
    ])

def p_groupby_basic_agg():
    return p_groupby([
        ("mean_close", pl.col("close").mean()),
        ("std_close", pl.col("close").std()),
        ("sum_volume", pl.col("volume").sum()),
        ("count_close", pl.col("close").count()),
    ])

def q_groupby_min_max():
    return q_groupby([
        ("min_low", col("low").min()),
        ("max_high", col("high").max()),
    ])

def p_groupby_min_max():
    return p_groupby([
        ("min_low", pl.col("low").min()),
        ("max_high", pl.col("high").max()),
    ])

def q_groupby_first_last():
    return q_groupby([
        ("first_open", col("open").first_value()),
        ("last_close", col("close").last_value()),
    ])

def p_groupby_first_last():
    return p_groupby([
        ("first_open", pl.col("open").first()),
        ("last_close", pl.col("close").last()),
    ])

def q_shift():
    return q_select([
        ("close_shift1", col("close").shift(1).expanding()),
    ], keys=("ticker", "datetime"))

def p_shift():
    return p_select([
        ("close_shift1", pl.col("close").shift(1)),
    ], keys=("ticker", "datetime"))

def q_ffill():
    return q_select([
        ("close_ffill", col("close").ffill().expanding()),
    ], keys=("ticker", "datetime"))

def p_ffill():
    return p_select([
        ("close_ffill", pl.col("close").forward_fill()),
    ], keys=("ticker", "datetime"))

def q_rolling_mean():
    return q_select([
        ("ma50", col("close").mean().rolling(50)),
    ])

def p_rolling_mean():
    return p_select([
        ("ma50", pl.col("close").rolling_mean(50)),
    ])

def q_rolling_std():
    return q_select([
        ("std50", col("close").std().rolling(50)),
    ])

def p_rolling_std():
    return p_select([
        ("std50", pl.col("close").rolling_std(50)),
    ])

def q_rolling_sum():
    return q_select([
        ("sum_volume50", col("volume").sum().rolling(50)),
    ])

def p_rolling_sum():
    return p_select([
        ("sum_volume50", pl.col("volume").rolling_sum(50)),
    ])

def q_rolling_rank():
    return q_select([
        ("rank50", col("close").rank().rolling(50)),
    ])

def p_rolling_rank():
    return p_select([
        ("rank50", pl.col("close").rolling_rank(50)),
    ])

def q_expanding_sum_over():
    return q_over([
        ("cum_close", col("close").sum().expanding()),
    ], "ticker", keys=("ticker", "datetime"))

def p_expanding_sum_over():
    return p_over([
        ("cum_close", pl.col("close").cum_sum()),
    ], "ticker", keys=("ticker", "datetime"))

def q_expanding_mean_over():
    return q_over([
        ("mean_close", col("close").mean().expanding()),
    ], "ticker", keys=("ticker", "datetime"))

def p_expanding_mean_over():
    return p_over([
        ("mean_close", pl.col("close").cum_sum() / pl.col("close").cum_count()),
    ], "ticker", keys=("ticker", "datetime"))

def q_rolling_mean_over():
    return q_over([
        ("ma50", col("close").mean().rolling(50)),
    ], "ticker", keys=("ticker", "datetime"))

def p_rolling_mean_over():
    return p_over([
        ("ma50", pl.col("close").rolling_mean(50)),
    ], "ticker", keys=("ticker", "datetime"))

def q_rolling_std_over():
    return q_over([
        ("std50", col("close").std().rolling(50)),
    ], "ticker", keys=("ticker", "datetime"))

def p_rolling_std_over():
    return p_over([
        ("std50", pl.col("close").rolling_std(50)),
    ], "ticker", keys=("ticker", "datetime"))

def q_rolling_rank_over():
    return q_over([
        ("rank50", col("close").rank().rolling(50)),
    ], "ticker", keys=("ticker", "datetime"))

def p_rolling_rank_over():
    return p_over([
        ("rank50", pl.col("close").rolling_rank(50)),
    ], "ticker", keys=("ticker", "datetime"))

def q_batch_rank_over_date():
    return q_over([
        ("rank_by_date", col("close").batch.rank()),
    ], "date", keys=("date", "ticker"))

def p_batch_rank_over_date():
    return p_over([
        ("rank_by_date", pl.col("close").rank()),
    ], "date", keys=("date", "ticker"))

def q_batch_sort():
    return col.all.batch.sort("close").calc_data(wide_sort_data)

def p_batch_sort():
    return wide_sort_data.sort("close")

def q_ta_rsi_over():
    return q_over([
        ("rsi14", col("close").ta.rsi(14).expanding()),
    ], "ticker", keys=("ticker", "datetime"))

duck_global_basic_agg_sql = duck_select([
    ("mean_close", "avg(close)"),
    ("std_close", "stddev_samp(close)"),
    ("sum_volume", "sum(volume)"),
    ("count_close", "count(close)"),
])
duck_global_min_max_sql = duck_select([
    ("min_low", "min(low)"),
    ("max_high", "max(high)"),
])
duck_groupby_basic_agg_sql = duck_select([
    ("mean_close", "avg(close)"),
    ("std_close", "stddev_samp(close)"),
    ("sum_volume", "sum(volume)"),
    ("count_close", "count(close)"),
], keys=("ticker",), group_by="ticker")
duck_groupby_min_max_sql = duck_select([
    ("min_low", "min(low)"),
    ("max_high", "max(high)"),
], keys=("ticker",), group_by="ticker")
duck_groupby_first_last_sql = duck_select([
    ("first_open", "first(open)"),
    ("last_close", "last(close)"),
], keys=("ticker",), group_by="ticker")
duck_shift_sql = duck_select([
    ("close_shift1", "lag(close, 1) over (order by datetime)"),
], keys=("ticker", "datetime"))
duck_rolling_mean_sql = duck_select([
    ("ma50", "avg(close) over (order by datetime rows between 49 preceding and current row)"),
])
duck_rolling_std_sql = duck_select([
    ("std50", "stddev_samp(close) over (order by datetime rows between 49 preceding and current row)"),
])
duck_rolling_sum_sql = duck_select([
    ("sum_volume50", "sum(volume) over (order by datetime rows between 49 preceding and current row)"),
])
duck_expanding_sum_over_sql = duck_select([
    ("cum_close", "sum(close) over (partition by ticker order by datetime rows between unbounded preceding and current row)"),
], keys=("ticker", "datetime"))
duck_rolling_mean_over_sql = duck_select([
    ("ma50", "avg(close) over (partition by ticker order by datetime rows between 49 preceding and current row)"),
], keys=("ticker", "datetime"))
duck_rolling_std_over_sql = duck_select([
    ("std50", "stddev_samp(close) over (partition by ticker order by datetime rows between 49 preceding and current row)"),
], keys=("ticker", "datetime"))
duck_batch_rank_sql = duck_select([
    ("rank_by_date", "rank() over (partition by \"date\" order by close)"),
], keys=("\"date\"", "ticker"))
duck_batch_sort_sql = "select * from wide_sort_data order by close"

scenarios = [
    Scenario("global_basic_agg", "global basic agg", "aggregation", "global", "mean/std/sum/count", "整列基础聚合。", q_global_basic_agg, p_global_basic_agg, duck_global_basic_agg_sql),
    Scenario("global_min_max", "global min/max", "aggregation", "global", "min/max", "整列 min/max。", q_global_min_max, p_global_min_max, duck_global_min_max_sql),
    Scenario("groupby_basic_agg", "group_by basic agg", "aggregation", "group_by(ticker)", "mean/std/sum/count", "按 ticker 分组基础聚合。", q_groupby_basic_agg, p_groupby_basic_agg, duck_groupby_basic_agg_sql),
    Scenario("groupby_min_max", "group_by min/max", "aggregation", "group_by(ticker)", "min/max", "按 ticker 分组 min/max。", q_groupby_min_max, p_groupby_min_max, duck_groupby_min_max_sql),
    Scenario("groupby_first_last", "group_by first/last", "aggregation", "group_by(ticker)", "first_value/last_value", "按 ticker 取第一行 open 和最后一行 close。", q_groupby_first_last, p_groupby_first_last, duck_groupby_first_last_sql),
    Scenario("shift_1", "shift", "series", "expanding", "shift(1)", "常见滞后一行。", q_shift, p_shift, duck_shift_sql),
    Scenario("ffill", "ffill", "series", "expanding", "ffill", "前值填充。", q_ffill, p_ffill, None),
    Scenario("rolling_mean_50", "rolling mean", "rolling", "rolling(50)", "mean", "50 窗口滚动均值。", q_rolling_mean, p_rolling_mean, duck_rolling_mean_sql),
    Scenario("rolling_std_50", "rolling std", "rolling", "rolling(50)", "std", "50 窗口滚动标准差。", q_rolling_std, p_rolling_std, duck_rolling_std_sql),
    Scenario("rolling_sum_50", "rolling sum", "rolling", "rolling(50)", "sum", "50 窗口滚动求和。", q_rolling_sum, p_rolling_sum, duck_rolling_sum_sql),
    Scenario("rolling_rank_50", "rolling rank", "rolling", "rolling(50)", "rank", "50 窗口滚动 rank。", q_rolling_rank, p_rolling_rank, None),
    Scenario("expanding_sum_over", "expanding sum over", "over/state", "expanding().over(ticker)", "sum", "每个 ticker 独立累计求和。", q_expanding_sum_over, p_expanding_sum_over, duck_expanding_sum_over_sql),
    Scenario("expanding_mean_over", "expanding mean over", "over/state", "expanding().over(ticker)", "mean", "每个 ticker 独立累计均值。", q_expanding_mean_over, p_expanding_mean_over, None),
    Scenario("rolling_mean_over", "rolling mean over", "over/state", "rolling(50).over(ticker)", "mean", "每个 ticker 独立 rolling mean。", q_rolling_mean_over, p_rolling_mean_over, duck_rolling_mean_over_sql),
    Scenario("rolling_std_over", "rolling std over", "over/state", "rolling(50).over(ticker)", "std", "每个 ticker 独立 rolling std。", q_rolling_std_over, p_rolling_std_over, duck_rolling_std_over_sql),
    Scenario("rolling_rank_over", "rolling rank over", "over/state", "rolling(50).over(ticker)", "rank", "每个 ticker 独立 rolling rank。", q_rolling_rank_over, p_rolling_rank_over, None),
    Scenario("batch_rank_date", "cross-section rank", "batch", "batch.over(date)", "batch.rank", "每个 date 横截面 rank。", q_batch_rank_over_date, p_batch_rank_over_date, duck_batch_rank_sql),
    Scenario("batch_sort_close", "sort", "batch", "batch", "sort", "按 close 排序。", q_batch_sort, p_batch_sort, duck_batch_sort_sql),
    Scenario("ta_rsi_over", "TA RSI", "domain", "expanding().over(ticker)", "rsi(14)", "常见技术指标 RSI，qust-only 参考。", q_ta_rsi_over, None, None),
]

pl.DataFrame([
    {
        "scenario": s.label,
        "family": s.family,
        "context": s.context,
        "operator": s.operator,
        "polars": s.polars_fn is not None,
        "qust": True,
        "duckdb": s.duckdb_sql is not None,
        "description": s.description,
    }
    for s in scenarios
])


scenario,family,context,operator,polars,qust,duckdb,description
str,str,str,str,bool,bool,bool,str
"""global basic agg""","""aggregation""","""global""","""mean/std/sum/count""",true,true,true,"""整列基础聚合。"""
"""global min/max""","""aggregation""","""global""","""min/max""",true,true,true,"""整列 min/max。"""
"""group_by basic agg""","""aggregation""","""group_by(ticker)""","""mean/std/sum/count""",true,true,true,"""按 ticker 分组基础聚合。"""
"""group_by min/max""","""aggregation""","""group_by(ticker)""","""min/max""",true,true,true,"""按 ticker 分组 min/max。"""
"""group_by first/last""","""aggregation""","""group_by(ticker)""","""first_value/last_value""",true,true,true,"""按 ticker 取第一行 open 和最后一行 close…"
"""shift""","""series""","""expanding""","""shift(1)""",true,true,true,"""常见滞后一行。"""
"""ffill""","""series""","""expanding""","""ffill""",true,true,false,"""前值填充。"""
"""rolling mean""","""rolling""","""rolling(50)""","""mean""",true,true,true,"""50 窗口滚动均值。"""
"""rolling std""","""rolling""","""rolling(50)""","""std""",true,true,true,"""50 窗口滚动标准差。"""


## 3. Benchmark 方法

每个 `engine/scenario` warmup 一次，然后重复 `REPEAT` 次。汇总使用 median 和 best。


In [13]:
def output_rows(obj) -> int:
    height = getattr(obj, "height", None)
    if height is not None:
        return int(height)
    shape = getattr(obj, "shape", None)
    if shape is not None:
        return int(shape[0])
    try:
        return int(len(obj))
    except Exception:
        return 0

def bench_one(engine: str, scenario: Scenario, fn: Callable[[], object], repeat: int = REPEAT, warmup: int = WARMUP) -> list[dict]:
    for _ in range(warmup):
        fn()
    rows = []
    for run in range(repeat):
        gc.collect()
        started = time.perf_counter()
        out = fn()
        elapsed_ms = (time.perf_counter() - started) * 1000.0
        rows.append({
            "scenario": scenario.name,
            "label": scenario.label,
            "family": scenario.family,
            "context": scenario.context,
            "operator": scenario.operator,
            "engine": engine,
            "run": run,
            "elapsed_ms": elapsed_ms,
            "rows": output_rows(out),
        })
    return rows

def make_duckdb_runner(sql: str):
    def run():
        result = duck_conn.execute(sql)
        if hasattr(result, "pl"):
            return result.pl()
        return result.fetchdf()
    return run


## 4. 执行 benchmark

Polars/qust/DuckDB 只在语义等价时对比。DuckDB 当前环境如果没安装，会自动跳过。


In [14]:
if HAS_DUCKDB:
    duck_conn = duckdb.connect(database=":memory:")

    def register_duck_frame(name: str, frame: pl.DataFrame):
        try:
            duck_conn.register(name, frame)
        except Exception:
            duck_conn.register(name, frame.to_arrow())

    register_duck_frame("bench_data", data)
    register_duck_frame("wide_sort_data", wide_sort_data)
else:
    duck_conn = None

bench_rows = []
for scenario in scenarios:
    if scenario.polars_fn is not None:
        bench_rows.extend(bench_one("polars", scenario, scenario.polars_fn))
    bench_rows.extend(bench_one("qust", scenario, scenario.qust_fn))
    if HAS_DUCKDB and scenario.duckdb_sql is not None:
        bench_rows.extend(bench_one("duckdb", scenario, make_duckdb_runner(scenario.duckdb_sql)))

runs = pl.DataFrame(bench_rows)
runs.head(18)


scenario,label,family,context,operator,engine,run,elapsed_ms,rows
str,str,str,str,str,str,i64,f64,i64
"""global_basic_agg""","""global basic agg""","""aggregation""","""global""","""mean/std/sum/count""","""polars""",0,3.736102,1
"""global_basic_agg""","""global basic agg""","""aggregation""","""global""","""mean/std/sum/count""","""polars""",1,3.103489,1
"""global_basic_agg""","""global basic agg""","""aggregation""","""global""","""mean/std/sum/count""","""polars""",2,3.656344,1
"""global_basic_agg""","""global basic agg""","""aggregation""","""global""","""mean/std/sum/count""","""polars""",3,2.335706,1
"""global_basic_agg""","""global basic agg""","""aggregation""","""global""","""mean/std/sum/count""","""polars""",4,3.592758,1
"""global_basic_agg""","""global basic agg""","""aggregation""","""global""","""mean/std/sum/count""","""qust""",0,13.498021,1
"""global_basic_agg""","""global basic agg""","""aggregation""","""global""","""mean/std/sum/count""","""qust""",1,12.565121,1
"""global_basic_agg""","""global basic agg""","""aggregation""","""global""","""mean/std/sum/count""","""qust""",2,17.518085,1
"""global_basic_agg""","""global basic agg""","""aggregation""","""global""","""mean/std/sum/count""","""qust""",3,21.852285,1


## 5. 汇总结果

`median_ms` 是主要读数，`best_ms` 是重复运行里最快的一次耗时。后面的柱状图都直接画实际耗时，单位是毫秒。


In [15]:
summary = (
    runs
    .group_by("scenario", "label", "family", "context", "operator", "engine")
    .agg(
        pl.col("elapsed_ms").median().alias("median_ms"),
        pl.col("elapsed_ms").min().alias("best_ms"),
        pl.col("rows").first().alias("rows"),
        pl.len().alias("runs"),
    )
)

polars_base = summary.filter(pl.col("engine") == "polars").select(
    "scenario",
    pl.col("median_ms").alias("polars_median_ms"),
)

summary = (
    summary
    .join(polars_base, on="scenario", how="left")
    .with_columns(
        pl.col("polars_median_ms").is_not_null().alias("has_polars_baseline"),
        (pl.col("rows") / (pl.col("median_ms") / 1000.0) / 1_000_000.0).alias("million_rows_per_sec"),
    )
)

scenario_order = {scenario.name: idx for idx, scenario in enumerate(scenarios)}
engine_order = {"polars": 0, "qust": 1, "duckdb": 2}

summary = pl.DataFrame([
    {
        **row,
        "scenario_order": scenario_order[row["scenario"]],
        "engine_order": engine_order.get(row["engine"], 99),
        "case_label": row["label"],
    }
    for row in summary.to_dicts()
]).sort("scenario_order", "engine_order")

summary.select(
    "family", "context", "operator", "engine",
    "median_ms", "best_ms", "rows", "runs", "million_rows_per_sec",
)


family,context,operator,engine,median_ms,best_ms,rows,runs,million_rows_per_sec
str,str,str,str,f64,f64,i64,i64,f64
"""aggregation""","""global""","""mean/std/sum/count""","""polars""",3.592758,2.335706,1,5,0.000278
"""aggregation""","""global""","""mean/std/sum/count""","""qust""",17.518085,12.565121,1,5,0.000057
"""aggregation""","""global""","""min/max""","""polars""",3.066436,2.678196,1,5,0.000326
"""aggregation""","""global""","""min/max""","""qust""",9.917763,8.880497,1,5,0.000101
"""aggregation""","""group_by(ticker)""","""mean/std/sum/count""","""polars""",16.504911,13.115799,4,5,0.000242
"""aggregation""","""group_by(ticker)""","""mean/std/sum/count""","""qust""",53.419717,50.901456,4,5,0.000075
"""aggregation""","""group_by(ticker)""","""min/max""","""polars""",9.367124,8.359393,4,5,0.000427
"""aggregation""","""group_by(ticker)""","""min/max""","""qust""",57.627689,56.318508,4,5,0.000069
"""aggregation""","""group_by(ticker)""","""first_value/last_value""","""polars""",8.563466,8.231376,4,5,0.000467


## 6. 不同引擎 median 耗时：并排多色柱

这个图使用宽表输入：第一列是 `case_label`，后续列分别是 `polars/qust/duckdb`。bar 底层会把多 y 序列横向错开，并用不同颜色显示。


In [16]:
def wide_metric(metric: str, source: pl.DataFrame, *, include_qust_only: bool = False) -> pl.DataFrame:
    frame = source
    if not include_qust_only:
        frame = frame.filter(pl.col("has_polars_baseline"))
    frame = frame.select(
        "scenario_order",
        "case_label",
        "engine",
        pl.col(metric).cast(pl.Float64).alias(metric),
    )
    if frame.height == 0:
        return pl.DataFrame({"case_label": []})
    wide = (
        frame
        .pivot(
            on="engine",
            index=["scenario_order", "case_label"],
            values=metric,
            aggregate_function="first",
        )
        .sort("scenario_order")
    )
    ordered_cols = [c for c in ["case_label", "polars", "qust", "duckdb"] if c in wide.columns]
    return wide.select(ordered_cols)

median_chart = wide_metric("median_ms", summary, include_qust_only=False)
median_chart = median_chart.select(
    "case_label",
    *[pl.col(c).round(4).alias(c) for c in median_chart.columns if c != "case_label"],
)

median_runtime = col(*median_chart.columns).monitor("median_ms_by_engine", show_axis_label=True).bar().runtime()
median_runtime.plot(median_chart, open_in_jupyter=True, auto_open=False, height=920)


## 7. 不同引擎 best 耗时：并排多色柱

第二张图仍然画真实耗时，单位是毫秒。这里取每个场景重复运行里的最快一次 `best_ms`，用来观察稳定后的执行开销。


In [ ]:
best_chart = wide_metric("best_ms", summary, include_qust_only=False)
best_chart = best_chart.select(
    "case_label",
    *[pl.col(c).round(4).alias(c) for c in best_chart.columns if c != "case_label"],
)

best_runtime = col(*best_chart.columns).monitor("best_ms_by_engine", show_axis_label=True).bar().runtime()
best_runtime.plot(best_chart, open_in_jupyter=True, auto_open=False, height=820)


## 8. qust-only 常见领域算子

这里保留常见技术指标 `RSI` 作为 qust-only 参考，用来展示领域状态算子的耗时。


In [18]:
qust_only_chart = (
    summary
    .filter((pl.col("engine") == "qust") & pl.col("polars_median_ms").is_null())
    .select(
        "case_label",
        pl.col("median_ms").round(4).alias("qust"),
    )
)

qust_only_runtime = col("case_label", "qust").monitor("qust_only_median_ms", show_axis_label=True).bar().runtime()
qust_only_runtime.plot(qust_only_chart, open_in_jupyter=True, auto_open=False, height=420)


## 9. 扩展示例的原则

新增性能场景时优先选择高频使用的算子，例如：聚合、rank、rolling、over、sort、shift、fill、常见 TA。只有确实需要展示某个专用能力时，才放 qust-only 场景。
